# GPT From Scratch by Rob

## Introduction
This section introduces the GPT-from-scratch implementation and outlines the main building blocks that will be defined in the notebook.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Tokenizing Input Text
This block prepares a small sample of text and encodes it with the GPT-2 tokenizer so the model can work with integer token IDs.

In [4]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Everyday holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109,  820, 6622,  257]])


In [13]:
GPT_CONFIG = {
    'vocab_size': 50257,
    'context_length': 1024,
    'emb_dim': 768,
    'n_heads': 12,
    'n_layers': 12,
    'drop_rate': 0.1,
    'qkv_bias': False
}


## Model Configuration
This block defines the core hyperparameters for the GPT model, including vocabulary size, context length, embedding size, and the number of layers and heads.

In [14]:
class LayerNorm (nn.Module):
    def __init__ (self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

## Layer Normalization
This block implements layer normalization, a technique used to stabilize training and improve convergence in deep neural networks.

In [15]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.44715 * torch.pow(x, 3))))
    

## Activation Function
This block defines the GELU activation function, which is commonly used in transformer models to improve training behavior.

In [16]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg['emb_dim'], 4 * cfg['emb_dim']),
            GELU(),
            nn.Linear(4 * cfg['emb_dim'], cfg['emb_dim']),
        )
    def forward(self, x):
        return self.layers(x)


## Feed-Forward Network
This block creates the feed-forward component of each transformer layer, which processes each token independently after attention.

In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out,
                 context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)
        
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32)), dim=-1)

        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ values).transpose(1, 2)
        
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )
        
        context_vec = self.out_proj(context_vec)
        return context_vec


## Multi-Head Attention
This block implements the attention mechanism that allows the model to focus on different parts of the input sequence simultaneously.

In [18]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in = cfg["emb_dim"],
            d_out = cfg["emb_dim"],
            context_length = cfg["context_length"],
            num_heads = cfg["n_heads"],
            dropout = cfg["drop_rate"],
            qkv_bias = cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        
    def forward(self, x):
        
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x
        

## Transformer Block
This block combines attention and feed-forward layers into a single transformer layer, with residual connections and normalization.

In [19]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False)
        
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device))
        
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


## Full GPT Model
This block assembles the complete GPT architecture by combining token embeddings, positional embeddings, transformer blocks, and the output head.

In [20]:
torch.manual_seed(42)
model = GPT(GPT_CONFIG)

out = model(batch)
print("Input batch:\n", batch)
print("Output shape:\n", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109,  820, 6622,  257]])
Output shape:
 torch.Size([2, 4, 50257])
tensor([[[-0.9734,  0.0339,  0.3249,  ..., -0.8772, -0.4005,  0.3409],
         [ 0.0693, -0.2877,  0.6123,  ...,  0.1722, -0.3115, -0.5939],
         [-0.7300,  0.3993,  0.2694,  ...,  1.0334,  0.1215, -0.4650],
         [ 0.0826,  0.6194,  0.1885,  ...,  0.3792, -1.1336, -0.0848]],

        [[-0.8342,  0.1903, -0.4076,  ..., -1.3926, -0.0823,  0.3429],
         [ 1.0269, -0.6121, -0.0960,  ...,  0.7721, -0.7334, -0.1049],
         [-0.3877,  0.9665,  0.6810,  ...,  1.2263,  0.0997,  1.2647],
         [-0.4795, -0.5068,  1.7054,  ...,  0.4517, -0.2336, -0.7341]]],
       grad_fn=<UnsafeViewBackward0>)


In [22]:
def generate(model, idx,
             max_new_token, context_size):
    for _ in range(max_new_token):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.argmax(probs, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

## Text Generation
This block defines the generation loop that repeatedly predicts the next token and appends it to the sequence.

In [23]:
start_context = "Hello i am"
encoded = tokenizer.encode(start_context)
print("encoded: ", encoded)
encoded_tensor = torch.tensor(encoded, dtype=torch.long).unsqueeze(0)
print("encoded_tensor shape: ", encoded_tensor.shape)

encoded:  [15496, 1312, 716]
encoded_tensor shape:  torch.Size([1, 3])


In [24]:
model.eval()
out = generate(
    model = model,
    idx = encoded_tensor,
    max_new_token = 6,
    context_size = GPT_CONFIG["context_length"]
)
print("output indices: ", out)
print("output length: ", len(out[0]))


output indices:  tensor([[15496,  1312,   716, 19840, 35886, 41900, 28324, 36068, 20012]])
output length:  9


In [25]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello i am Graphics Null spa499 pilgrimageCommunity


## Decoding the Output
This final block decodes the generated token IDs back into human-readable text so the model output can be inspected.